# Carpet WaveToy Thorns

Generate Carpet-style WaveToy thorns and inspect the Cactus Configuration
Language (CCL) and source files they provide.

Navigation:
[Index](../index.ipynb) |
Previous: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) |
Next: [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)

## Learning Goals

- Generate a small set of Einstein Toolkit thorns for the wave equation.
- Separate evolution, initial-data, and diagnostic roles.
- Read schedule entries as the order of operations in a Toolkit run.

## Words for This Notebook

- **Carpet:** an Einstein Toolkit mesh driver used by older Toolkit examples.
- **WaveToy:** a simple wave-equation example used by Toolkit tutorials.
- **Thorn:** an Einstein Toolkit component folder.
- **CCL file:** a text file that declares thorn metadata.
- **Evolution thorn:** the component that updates fields in time.
- **Initial-data thorn:** the component that sets starting field values.
- **Diagnostic thorn:** the component that writes numbers used to check the run.
- **Installed package:** a Python package imported from `site-packages`.

Use the code cells actively: first predict what should happen, then run the cell.
Afterward, explain the output in plain language.

## Table of Contents

- [Generated Thorn Roles](#Generated-Thorn-Roles)
- [Step 1: Import Thorn Inspection Helpers](#Step-1:-Import-Thorn-Inspection-Helpers)
- [Step 2: Create a Carpet Thorn Workspace](#Step-2:-Create-a-Carpet-Thorn-Workspace)
- [Step 3: Generate Carpet-Compatible Thorns](#Step-3:-Generate-Carpet-Compatible-Thorns)
- [Learning Check](#Learning-Check)
- [Step 4: Inspect the Generated Inventory](#Step-4:-Inspect-the-Generated-Inventory)
- [Validation Check](#Validation-Check)
- [Step 5: Inspect Evolution Variables](#Step-5:-Inspect-Evolution-Variables)
- [Step 6: Read the Evolution Schedule](#Step-6:-Read-the-Evolution-Schedule)
- [Step 7: Inspect Initial-Data Variables](#Step-7:-Inspect-Initial-Data-Variables)
- [Step 8: Read the Diagnostics Schedule](#Step-8:-Read-the-Diagnostics-Schedule)
- [Step 9: Inspect One Generated Source File](#Step-9:-Inspect-One-Generated-Source-File)

## Generated Thorn Roles

| Thorn | Role | Files to inspect |
| --- | --- | --- |
| `WaveToyNRPy` | Evolves wave fields | `interface.ccl`, `schedule.ccl`, RHS source |
| `IDWaveToyNRPy` | Sets initial data | `interface.ccl`, exact-solution source |
| `diagWaveToyNRPy` | Writes diagnostics | `schedule.ccl`, exact-solution source |

## Step 1: Import Thorn Inspection Helpers

These standard-library tools run commands, manage temporary project directories,
clean command output, and reject local NRPy source-tree imports.

If you are new to Python, skim this helper cell on a first pass. Its job is to
run terminal commands, shorten command output, and stop with a clear message if
a required command is missing.

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def local_nrpy_source_roots():
    roots = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "nrpy"
        if (candidate / "nrpy" / "__init__.py").is_file():
            roots.append(candidate.resolve())
    return roots


def require_installed_nrpy(import_path):
    resolved = Path(import_path).resolve()
    for source_root in local_nrpy_source_roots():
        if resolved.is_relative_to(source_root):
            raise RuntimeError(f"NRPy loaded from local source tree: {resolved}")
    installed_markers = {"site-packages", "dist-packages"}
    if not installed_markers.intersection(resolved.parts):
        raise RuntimeError(f"NRPy did not load from an installed package: {resolved}")
    return resolved


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            env=os.environ.copy(),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)

## Step 2: Create a Carpet Thorn Workspace

The workspace keeps generated files separate from the tutorial source tree.

In [2]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_carpet_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "et_wavetoy"

print("workspace:", WORKSPACE)
print("project path:", PROJECT_DIR)

workspace: /work/5-infrastructures/nrpy_tutorial_carpet_eqt7m02z
project path: /work/5-infrastructures/nrpy_tutorial_carpet_eqt7m02z/project/et_wavetoy


## Step 3: Generate Carpet-Compatible Thorns

Compiling or running these thorns requires an external Einstein Toolkit checkout.
This notebook focuses on generation and inspection.

In [3]:
check_command = [
    sys.executable,
    "-c",
    "import nrpy, pathlib; print(pathlib.Path(nrpy.__file__).resolve())",
]
nrpy_import_path = Path(run_command(check_command, WORKSPACE, 30).strip())
nrpy_import_path = require_installed_nrpy(nrpy_import_path)
print("subprocess NRPy installed package:", nrpy_import_path)

command = [sys.executable, "-m", "nrpy.examples.carpet_wavetoy_thorns"]
print("generator command: python -m nrpy.examples.carpet_wavetoy_thorns")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project generated:", PROJECT_DIR.relative_to(WORKSPACE))

subprocess NRPy installed package: /virt/lib/python3.12/site-packages/nrpy/__init__.py
generator command: python -m nrpy.examples.carpet_wavetoy_thorns


In 0.094s, worker completed task 'register_CFunction_rhs_eval'
In 0.117s, worker completed task 'register_CFunction_exact_solution_all_points'
In 0.118s, worker completed task 'register_CFunction_exact_solution_all_points'
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/IDWaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/schedule.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/interface.ccl...[written]
Checking <workspace>/project/et_wavetoy/diagWaveToyNRPy/param.ccl...[written]
Checking <workspace>/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_MoL_

## Learning Check

Before inspecting the inventory, predict why the project needs more than one
thorn. After inspection, assign each thorn a role in one sentence.

## Step 4: Inspect the Generated Inventory

The inventory identifies the generated files relevant to this lesson.

In [4]:
print("thorn inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

thorn inventory:
IDWaveToyNRPy/interface.ccl
IDWaveToyNRPy/param.ccl
IDWaveToyNRPy/schedule.ccl
IDWaveToyNRPy/src/IDWaveToyNRPy_exact_solution_all_points.c
IDWaveToyNRPy/src/make.code.defn
WaveToyNRPy/interface.ccl
WaveToyNRPy/param.ccl
WaveToyNRPy/schedule.ccl
WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c
WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c
WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c
WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c
WaveToyNRPy/src/WaveToyNRPy_specify_aux_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_evol_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_zero_rhss.c
WaveToyNRPy/src/make.code.defn
WaveToyNRPy/src/simd/simd_intrinsics.h
diagWaveToyNRPy/interface.ccl
diagWaveToyNRPy/param.ccl
diagWaveToyNRPy/schedule.ccl
diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c
diagWaveToyNRPy/src/make.code.defn


## Validation Check

Verify that the generated Carpet project contains the thorns and files that this
notebook inspects.

In [5]:
required_files = [
    "WaveToyNRPy/interface.ccl",
    "WaveToyNRPy/schedule.ccl",
    "WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c",
    "IDWaveToyNRPy/interface.ccl",
    "IDWaveToyNRPy/schedule.ccl",
    "diagWaveToyNRPy/interface.ccl",
    "diagWaveToyNRPy/schedule.ccl",
    "diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c",
]
missing = [
    relative_path
    for relative_path in required_files
    if not (PROJECT_DIR / relative_path).is_file()
]
if missing:
    raise FileNotFoundError("Missing generated files: " + ", ".join(missing))
print("validation passed: required Carpet thorn files are present")

validation passed: required Carpet thorn files are present


## Step 5: Inspect Evolution Variables

The `interface.ccl` file declares the evolved variables and inherited Toolkit
capabilities.

In [6]:
relative_path = "WaveToyNRPy/interface.ccl"
evolution_interface_text = (PROJECT_DIR / relative_path).read_text(
    encoding="utf-8", errors="replace"
)
print(f"--- {relative_path} ---")
print(evolution_interface_text)

--- WaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: WaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION S

## Step 6: Read the Evolution Schedule

The full schedule file shows where the right-hand side and boundary hooks enter
the Toolkit time step. Look for schedule bins, reads, writes, and function names.

In [7]:
relative_path = "WaveToyNRPy/schedule.ccl"
evolution_schedule_text = (PROJECT_DIR / relative_path).read_text(
    encoding="utf-8", errors="replace"
)
print(f"--- {relative_path} ---")
print(evolution_schedule_text)

--- WaveToyNRPy/schedule.ccl ---
# This schedule.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

##################################################
# Step 0: Allocate memory for gridfunctions, using the STORAGE: keyword.

STORAGE: evol_variables[3]     # Evolution variables
STORAGE: evol_variables_rhs[1] # Variables storing right-hand-sides
STORAGE: aux_variables[3]      # Diagnostics variables


##################################################
# Step 1: Schedule functions in the Driver_BoundarySelect scheduling bin.

schedule WaveToyNRPy_specify_Driver_BoundaryConditions in Driver_BoundarySelect
{
  LANG: C
  OPTIONS: meta
} "Register boundary conditions in PreSync bin Driver_BoundarySelect."

##################################################
# Step 2: Schedule functions in the BASEGRID scheduling bin.

schedule WaveToyNRPy_Symmetry_registration_oldCartGrid3D at BASEGRID as Symme

## Step 7: Inspect Initial-Data Variables

The initial-data interface declares the fields initialized before time evolution
begins.

In [8]:
relative_path = "IDWaveToyNRPy/interface.ccl"
initial_data_interface_text = (PROJECT_DIR / relative_path).read_text(
    encoding="utf-8", errors="replace"
)
print(f"--- {relative_path} ---")
print(initial_data_interface_text)

--- IDWaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: IDWaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Grid WaveToyNRPy # WaveToyNRPy provides all gridfunctions.

# Needed functions and #include's:


# FIXME: add info for symmetry conditions:
#    https://einsteintoolkit.org/thornguide/CactusBase/SymBase/documentation.html

# Tell the Toolkit that we want all gridfunctions
#    to be visible to other thorns by using
#    the keyword "public". Note that declaring these
#    gridfunctions *does not* allocate memory for them;
#    that is done by the schedule.ccl file.

public:



## Step 8: Read the Diagnostics Schedule

The full diagnostics schedule shows where output is written during the run. Look
for the scheduled function and the bin that controls when diagnostics run.

In [9]:
relative_path = "diagWaveToyNRPy/schedule.ccl"
diagnostics_schedule_text = (PROJECT_DIR / relative_path).read_text(
    encoding="utf-8", errors="replace"
)
print(f"--- {relative_path} ---")
print(diagnostics_schedule_text)

--- diagWaveToyNRPy/schedule.ccl ---
# This schedule.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

##################################################
# Step 0: Allocate memory for gridfunctions, using the STORAGE: keyword.

STORAGE: aux_variables[3] # Diagnostics variables


##################################################
# Step 1: Schedule functions in the remaining scheduling bins.

schedule diagWaveToyNRPy_exact_solution_all_points IN CCTK_ANALYSIS
{
  LANG: C
  READS: Grid::x(Everywhere)
  READS: Grid::y(Everywhere)
  READS: Grid::z(Everywhere)
  WRITES: WaveToyNRPy::uu_exactGF(Everywhere)
  WRITES: WaveToyNRPy::vv_exactGF(Everywhere)
} "Set up metric fields for binary black hole initial data"



## Step 9: Inspect One Generated Source File

The full right-hand-side source file represents the generated C output path. Look
for the scheduled function name, the grid loop, and the right-hand-side
assignment.

In [10]:
source_path = PROJECT_DIR / "WaveToyNRPy" / "src" / "WaveToyNRPy_rhs_eval.c"
if not source_path.is_file():
    raise FileNotFoundError(source_path)
rhs_source_text = source_path.read_text(encoding="utf-8", errors="replace")
print(rhs_source_text)

#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"
#include "math.h"
#include "simd/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void WaveToyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_WaveToyNRPy_rhs_eval;
  const CCTK_REAL *restrict NOSIMDwavespeed = CCTK_ParameterGet("wavespeed", "IDWaveToyNRPy", NULL); // IDWaveToyNRPy::wavespeed
  const REAL_SIMD_ARRAY wavespeed = ConstSIMD(*NOSIMDwavespeed);                                     // IDWaveToyNRPy::wavespeed
  const CCTK_REAL NOSIMDinvdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const REAL_SIMD_ARRAY invdxx0 = ConstSIMD(NOSIMDinvdxx0);
  const CCTK_REAL NOSIMDinvdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const REAL_SIMD_ARRAY invdxx1 = ConstSIMD(NOSIMDinvdxx1);
  const CCTK_REAL NOSIMDinvdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
  const REAL_SIMD_ARRAY invdxx2 = ConstSIMD(NOSIMDinvdxx2);
#pragma omp parallel for
  for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2] - cctk_nghostzones[2]; i2++) {
    for (in

The thorn inventory, CCL files, and full generated source file show the three
pieces of the Carpet WaveToy example: evolution, initial data, and diagnostics.

## Continue to Code-Writing Choices

- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)